In [ ]:
# Upload Dataset File
from google.colab import files
uploaded = files.upload()

import pandas as pd

filename = list(uploaded.keys())[0]

if filename.endswith('.csv'):
    df = pd.read_csv(filename)
elif filename.endswith('.xlsx'):
    df = pd.read_excel(filename)
else:
    raise ValueError("Unsupported file type")


Saving Cleaned_20240320.xlsx to Cleaned_20240320 (2).xlsx


In [ ]:
# Loads Excel file into a DataFrame (like a table)
filename = list(uploaded.keys())[0]

# Auto-detect file type
if filename.endswith('.csv'):
    df = pd.read_csv(filename)
elif filename.endswith('.xlsx') or filename.endswith('.xls'):
    df = pd.read_excel(filename)
else:
    df = pd.read_csv(filename, sep=None, engine='python')

# Shows the first 5 rows- to check if it loaded correctly.
df.head()

target_col = 'Risk Level'

# Normalize
df[target_col] = df[target_col].astype(str).str.strip().str.lower()

# Map
risk_order = {
    "low": 0,
    "moderate": 1,
    "high": 2
}

df[target_col] = df[target_col].map(risk_order)

# Remove invalid/unmapped rows
df = df[df[target_col].notna()].copy()

# Check
print("Unique after mapping:", df[target_col].unique())

Unique after mapping: [1. 0. 2.]


In [ ]:
# LOAD TRAIN-TEST DATA

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob
import os
import joblib

# Locate latest processed dataset folder
folders = glob.glob("/content/drive/MyDrive/data_after_corr_*")

if len(folders) == 0:
    raise ValueError(
        "No saved dataset found."
    )

latest_folder = max(
    folders,
    key=os.path.getmtime
)

print("Loading data from:")
print(latest_folder)

# Load EXACT SAME train-test split
X_train = joblib.load(
    os.path.join(
        latest_folder,
        "X_train_final.pkl"
    )
)

X_test = joblib.load(
    os.path.join(
        latest_folder,
        "X_test_final.pkl"
    )
)

y_train = joblib.load(
    os.path.join(
        latest_folder,
        "y_train.pkl"
    )
)

y_test = joblib.load(
    os.path.join(
        latest_folder,
        "y_test.pkl"
    )
)

print("\nLoaded Successfully")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Mounted at /content/drive
Loading data from:
/content/drive/MyDrive/data_after_corr_20260507_1258

Loaded Successfully
X_train: (264, 42)
X_test: (66, 42)
y_train: (264,)
y_test: (66,)


In [ ]:
# CREATE FEATURE-SELECTED DATA

# Check missing features FIRST
missing = [
    f for f in FEATURES
    if f not in X_train.columns
]

print("\nMissing features:")
print(missing)

# STOP if features missing
if len(missing) > 0:

    print("\nAvailable columns:")
    print(X_train.columns.tolist())

    raise ValueError(
        "Some selected features do not exist in X_train."
    )

# USE EXACT ORIGINAL FEATURES
X_train_sel = X_train[FEATURES].copy()
X_test_sel = X_test[FEATURES].copy()

print("\nTrain Shape:", X_train_sel.shape)
print("Test Shape:", X_test_sel.shape)

# =========================================
# FINAL CLEANING
# =========================================

# Fill NaN ONLY
X_train_sel = X_train_sel.fillna(-999)
X_test_sel = X_test_sel.fillna(-999)

print("\nNaN in X_train_sel:",
      X_train_sel.isna().sum().sum())

print("NaN in X_test_sel:",
      X_test_sel.isna().sum().sum())

# =========================================
# EVALUATION FUNCTIONS
# =========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate(y_test, y_pred):

    return {

        "Accuracy":
        accuracy_score(
            y_test,
            y_pred
        ) * 100,

        "Precision (W)":
        precision_score(
            y_test,
            y_pred,
            average='weighted',
            zero_division=0
        ) * 100,

        "Recall (W)":
        recall_score(
            y_test,
            y_pred,
            average='weighted',
            zero_division=0
        ) * 100,

        "F1 (W)":
        f1_score(
            y_test,
            y_pred,
            average='weighted',
            zero_division=0
        ) * 100,

        "Precision (Macro)":
        precision_score(
            y_test,
            y_pred,
            average='macro',
            zero_division=0
        ) * 100,

        "Recall (Macro)":
        recall_score(
            y_test,
            y_pred,
            average='macro',
            zero_division=0
        ) * 100,

        "F1 (Macro)":
        f1_score(
            y_test,
            y_pred,
            average='macro',
            zero_division=0
        ) * 100,
    }

def print_results(results):

    for k, v in results.items():

        print(f"{k}: {v:.2f}%")

# =========================================
# DECISION TREE MODEL
# =========================================

import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

# SAME CV AS ORIGINAL
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# SAME MODEL AS ORIGINAL
dt_model = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42
)

# SAME PARAM GRID
param_grid = {
    "max_depth": list(range(2, 30))
}

# SAME RANDOM SEARCH
total_space = np.prod(
    [len(v) for v in param_grid.values()]
)

n_iter = min(10, total_space)

search = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=param_grid,
    n_iter=n_iter,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

# TRAIN
search.fit(
    X_train_sel,
    y_train
)

print("\nBest Params:")
print(search.best_params_)

best_model = search.best_estimator_

# TEST
y_pred = search.predict(X_test_sel)

# RESULTS
print("\n========== DECISION TREE RESULTS ==========")

print_results(
    evaluate(y_test, y_pred)
)


Missing features:
[]

Train Shape: (264, 14)
Test Shape: (66, 14)

NaN in X_train_sel: 0
NaN in X_test_sel: 0

Best Params:
{'max_depth': 27}

========== DECISION TREE RESULTS ==========
Accuracy: 90.91%
Precision (W): 90.91%
Recall (W): 90.91%
F1 (W): 90.91%
Precision (Macro): 83.59%
Recall (Macro): 83.59%
F1 (Macro): 83.59%


In [ ]:
# ADD PREDICTIONS TO DATASET

df["Predicted Risk Level"] = predicted_labels

df["Predicted Risk Level Encoded"] = (
    df["Predicted Risk Level"]
    .str.lower()
    .map({
        "low": 0,
        "moderate": 1,
        "high": 2
    })
)
# =========================================
# SAVE FINAL DATASET
# =========================================

from google.colab import drive

drive.mount('/content/drive')

output_path = (
    "/content/drive/MyDrive/"
    "dataset_with_predicted_risk.csv"
)

df.to_csv(
    output_path,
    index=False
)

print("\nSaved to:")
print(output_path)
# =========================================
# SAVE MODEL
# =========================================

model_path = (
    "/content/drive/MyDrive/"
    "decision_tree_model.pkl"
)

joblib.dump(
    best_model,
    model_path
)

print("\nModel saved:")
print(model_path)

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

filename = list(uploaded.keys())[0]

if filename.endswith('.csv'):
    df_raw = pd.read_csv(filename)
elif filename.endswith('.xlsx'):
    df_raw = pd.read_excel(filename)
else:
    raise ValueError("Unsupported file type")

print("Raw dataset shape:", df_raw.shape)


In [ ]:
# Replace column of risk level in raw dataset
X_full = df_raw[FEATURES].copy()

X_full = X_full.fillna(-999)

for col in cat_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].astype(str)

y_pred_full = best_model.predict(X_full).flatten()

print(len(y_pred_full), len(df_raw))   # should be 436, 436

df_raw["Predicted Risk Level"] = [
    reverse_map[int(i)] for i in y_pred_full
]
risk_map = {
    "low": 0,
    "moderate": 1,
    "high": 2
}

df_raw["Predicted Risk Level Encoded"] = (
    df_raw["Predicted Risk Level"].str.lower().map(risk_map)
)

In [ ]:
# Save
from google.colab import drive
drive.mount('/content/drive')

output_path = "/content/drive/MyDrive/dataset_with_predicted_risk.csv"

df_raw.to_csv(output_path, index=False)

print("Saved to:", output_path)

import os
print(os.listdir("/content/drive/MyDrive"))